# Integrated OSCC Graph-RAG Research Application

Combines the existing trained EfficientNetB0 classifier, four attribution methods (Grad-CAM, Grad-CAM++, LayerCAM, HiResCAM), Neo4j, the existing FAISS literature index and Sentence Transformer, Groq report generation, FastAPI, and the embedded frontend.

The classifier is inference-only. It is loaded from the configured Google Drive path, uses the fixed OSCC threshold 0.850266, and is never trained or updated. Model output and heatmaps are research signals, not a clinical diagnosis or proof of pathology.


In [ ]:
!pip -q install fastapi uvicorn python-multipart pyngrok neo4j faiss-cpu sentence-transformers groq requests pillow


In [ ]:
from google.colab import drive, userdata
import os, json, uuid, time, threading, subprocess
from datetime import datetime, timezone
from pathlib import Path

drive.mount('/content/drive')
def get_secret(name):
    value=userdata.get(name)
    if not value: raise RuntimeError(f'Missing Colab Secret: {name}')
    return value
NEO4J_URI=get_secret('NEO4J_URI')
NEO4J_USERNAME=get_secret('NEO4J_USERNAME')
NEO4J_PASSWORD=get_secret('NEO4J_PASSWORD')
NEO4J_DATABASE=get_secret('NEO4J_DATABASE')
GROQ_API_KEY=get_secret('GROQ_API_KEY')
NGROK_AUTHTOKEN=get_secret('NGROK_AUTHTOKEN')
MODEL_PATH='/content/drive/MyDrive/OSCC V2/best_efficientnetb0.keras'
FAISS_PATH='/content/drive/MyDrive/OSCC_Project/graphrag_phase2/oscc_literature.faiss'
METADATA_PATH='/content/drive/MyDrive/OSCC_Project/graphrag_phase2/chunk_metadata.json'
ARTIFACT_DIR=Path('/content/oscc_graphrag_artifacts')
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
CLASSIFICATION_THRESHOLD=0.850266
CLASS_MAPPING={'Normal':0,'OSCC':1}
print('Configuration loaded from Colab Secrets and Drive paths.')


In [ ]:
import tensorflow as tf
import numpy as np
model=tf.keras.models.load_model(MODEL_PATH,compile=False)
assert tuple(model.input_shape[1:])==(224,224,3), model.input_shape
assert tuple(model.output_shape[1:])==(1,), model.output_shape
print('Existing EfficientNetB0 loaded for inference:',MODEL_PATH)
print('Input:',model.input_shape,'Output:',model.output_shape)


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
faiss_index=faiss.read_index(FAISS_PATH)
with open(METADATA_PATH,'r',encoding='utf-8') as f: chunk_metadata=json.load(f)
embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('FAISS vectors:',faiss_index.ntotal,'metadata records:',len(chunk_metadata))


In [ ]:
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    session.run("RETURN 1 AS ok").single()

print("Neo4j connected.")

In [ ]:
from groq import Groq
groq_client=Groq(api_key=GROQ_API_KEY)
GROQ_MODEL='openai/gpt-oss-20b'
print('Groq client initialized.')


In [ ]:
import re
import numpy as np
import tensorflow as tf
from PIL import Image

IMAGE_SIZE=(224,224)
XAI_METHOD_NAMES=['Grad-CAM','Grad-CAM++','LayerCAM','HiResCAM']
XAI_FILE_NAMES={'Grad-CAM':'grad_cam','Grad-CAM++':'grad_cam_plus_plus','LayerCAM':'layer_cam','HiResCAM':'hires_cam'}

def preprocess_image_bytes(image_bytes):
    decoded=tf.io.decode_image(tf.convert_to_tensor(image_bytes),channels=3,expand_animations=False)
    decoded.set_shape([None,None,3])
    resized=tf.image.resize(decoded,IMAGE_SIZE,antialias=True)
    resized=tf.cast(resized,tf.float32)  # retain the 0-255 range
    resized=tf.clip_by_value(
        resized,
        clip_value_min=0.0,
        clip_value_max=255.0
    )
    tf.debugging.assert_all_finite(resized,'Non-finite image input')
    tf.debugging.assert_greater_equal(tf.reduce_min(resized),0.)
    tf.debugging.assert_less_equal(tf.reduce_max(resized),255.)
    model_input=tf.keras.applications.efficientnet.preprocess_input(resized)
    return model_input[None,...],np.clip(resized.numpy(),0,255).astype(np.uint8)

def predict_image(image_bytes):
    image_tensor,rgb_image=preprocess_image_bytes(image_bytes)
    probability=float(model(image_tensor,training=False).numpy().reshape(-1)[0])
    label='OSCC' if probability>=CLASSIFICATION_THRESHOLD else 'Normal'
    class_confidence=probability if label=='OSCC' else 1.-probability
    return {'image_tensor':image_tensor,'rgb_image':rgb_image,'oscc_probability':probability,'prediction':label,'class_index':CLASS_MAPPING[label],'confidence':float(class_confidence)}

def inference_call(layer,x):
    try: return layer(x,training=False)
    except TypeError: return layer(x)

# Reuse the original nested EfficientNet and head layers. Replay the outer prefix
# into the backbone probe to keep tensors graph-connected (no disconnected prefix model).
backbone=model.get_layer('efficientnetb0')
last_conv=backbone.get_layer('top_conv')
assert tuple(backbone.output_shape[1:])==(7,7,1280),backbone.output_shape
assert tuple(last_conv.output.shape[1:])==(7,7,1280),last_conv.output.shape
backbone_index=model.layers.index(backbone)
head_layers=model.layers[backbone_index+1:-1]
output_layer=model.layers[-1]
allowed_head=(tf.keras.layers.GlobalAveragePooling2D,tf.keras.layers.GlobalMaxPooling2D,tf.keras.layers.Flatten,tf.keras.layers.BatchNormalization,tf.keras.layers.Dropout,tf.keras.layers.Dense,tf.keras.layers.ReLU,tf.keras.layers.Activation)
unsupported=[(l.name,type(l).__name__) for l in head_layers if not isinstance(l,allowed_head)]
assert not unsupported,f'Unsupported classifier head for analytic Grad-CAM++: {unsupported}'
for layer in head_layers:
    if isinstance(layer,(tf.keras.layers.Dense,tf.keras.layers.Activation)):
        assert tf.keras.activations.serialize(layer.activation) in ('linear','relu'),layer.name
assert isinstance(output_layer,tf.keras.layers.Dense) and output_layer.units==1
assert tf.keras.activations.serialize(output_layer.activation)=='sigmoid'
conv_probe=tf.keras.Model(backbone.input,[last_conv.output,backbone.output],name='trained_efficientnetb0_conv_probe')

def run_model_prefix(x):
    z=x
    for layer in model.layers[1:backbone_index]: z=inference_call(layer,z)
    return z

def head_logits_from_backbone_output(backbone_output):
    z=backbone_output
    for layer in head_layers: z=inference_call(layer,z)
    if output_layer.use_bias: return tf.linalg.matmul(z,output_layer.kernel)+output_layer.bias
    return tf.linalg.matmul(z,output_layer.kernel)

def resize_normalize(raw):
    heat=tf.image.resize(raw[None,...,None],IMAGE_SIZE,method='bilinear',antialias=False)[0,...,0]
    heat=tf.where(tf.math.is_finite(heat),heat,tf.zeros_like(heat))
    lo=tf.reduce_min(heat); hi=tf.reduce_max(heat)
    return tf.math.divide_no_nan(heat-lo,hi-lo)

def heatmap_similarity(maps):
    names=list(maps); result={}
    for i,a_name in enumerate(names):
        for b_name in names[i+1:]:
            a=maps[a_name].reshape(-1).astype(np.float64); b=maps[b_name].reshape(-1).astype(np.float64)
            ac=a-a.mean(); bc=b-b.mean(); denom=float(np.linalg.norm(ac)*np.linalg.norm(bc))
            cden=float(np.linalg.norm(a)*np.linalg.norm(b))
            result[f'{a_name} vs {b_name}']={'pearson_correlation':float(np.dot(ac,bc)/denom) if denom else None,'cosine_similarity':float(np.dot(a,b)/cden) if cden else None}
    return result

def failed_method(exc):
    return {'status':'failed','error':type(exc).__name__,'heatmap_path':None,'heatmap_url':None,'overlay_path':None,'overlay_url':None,'normalization':'Per-image min-max to [0,1]; constant maps are zero.'}

def generate_xai(image_tensor,rgb_image,case_id,expected_probability):
    methods={name:{'status':'failed','error':'shared_gradient_extraction_failed'} for name in XAI_METHOD_NAMES}
    try:
        # One feature extraction and one gradient calculation serve all four methods.
        with tf.GradientTape(persistent=True) as tape:
            prefix=run_model_prefix(image_tensor)
            activations,backbone_output=conv_probe(prefix,training=False)
            logits=head_logits_from_backbone_output(backbone_output)
            score=tf.math.sigmoid(logits)
        score_gradient=tape.gradient(score,activations)
        logit_gradient=tape.gradient(logits,activations)
        del tape
        probe_probability=float(score.numpy().reshape(-1)[0])
        np.testing.assert_allclose(probe_probability,expected_probability,rtol=2e-4,atol=2e-5)
        A=activations[0]; G=score_gradient[0]; GL=logit_gradient[0]
        p=tf.clip_by_value(tf.reshape(score,[]),tf.keras.backend.epsilon(),1.-tf.keras.backend.epsilon())
        s1=p*(1.-p); s2=s1*(1.-2.*p); s3=s1*(1.-6.*p+6.*tf.square(p))
        d1=s1*GL; d2=s2*tf.square(GL); d3=s3*tf.pow(GL,3)
        sum_A=tf.reduce_sum(A,axis=[0,1],keepdims=True)
        alpha_pp=tf.math.divide_no_nan(d2,2.*d2+sum_A*d3)
        weights_pp=tf.reduce_sum(alpha_pp*tf.nn.relu(d1),axis=[0,1],keepdims=True)
        raw_methods={
            'Grad-CAM':lambda:tf.nn.relu(tf.reduce_sum(A*tf.reduce_mean(G,axis=[0,1],keepdims=True),axis=-1)),
            'Grad-CAM++':lambda:tf.nn.relu(tf.reduce_sum(weights_pp*A,axis=-1)),
            'LayerCAM':lambda:tf.nn.relu(tf.reduce_sum(A*tf.nn.relu(G),axis=-1)),
            'HiResCAM':lambda:tf.nn.relu(tf.reduce_sum(A*G,axis=-1)),
        }
        maps={}; case_dir=ARTIFACT_DIR/case_id; case_dir.mkdir(parents=True,exist_ok=True)
        for method in XAI_METHOD_NAMES:
            try:
                normalized=resize_normalize(raw_methods[method]()).numpy()
                if not np.isfinite(normalized).all(): raise ValueError('non_finite_heatmap')
                maps[method]=normalized
                stats={'min':float(normalized.min()),'max':float(normalized.max()),'mean':float(normalized.mean()),'std':float(normalized.std()),'q90':float(np.quantile(normalized,.90)),'fraction_at_or_above_0_5':float(np.mean(normalized>=.5))}
                stem=XAI_FILE_NAMES[method]; heat_name=stem+'.png'; overlay_name=stem+'_overlay.png'
                heat8=np.clip(normalized*255.,0,255).astype(np.uint8)
                Image.fromarray(heat8,mode='L').save(case_dir/heat_name)
                red=np.zeros_like(rgb_image); red[...,0]=heat8
                overlay=np.clip(.58*rgb_image.astype(np.float32)+.42*red.astype(np.float32),0,255).astype(np.uint8)
                Image.fromarray(overlay,mode='RGB').save(case_dir/overlay_name)
                methods[method]={'status':'ok','heatmap_path':str(case_dir/heat_name),'heatmap_url':f'/heatmaps/{case_id}/{heat_name}','overlay_path':str(case_dir/overlay_name),'overlay_url':f'/heatmaps/{case_id}/{overlay_name}','normalization':'Per-image min-max to [0,1] after method ReLU; constant maps are zero.','statistics':stats}
            except Exception as exc:
                methods[method]=failed_method(exc)
        return {'target':'continuous OSCC sigmoid score','prediction_equivalence_verified':True,'methods':methods,'pairwise_similarity':heatmap_similarity(maps),'scope_note':'Attribution only; not lesion detection, pathology confirmation, or evidence of cancer.'}
    except Exception as exc:
        return {'target':'continuous OSCC sigmoid score','prediction_equivalence_verified':False,'methods':{name:failed_method(exc) for name in XAI_METHOD_NAMES},'pairwise_similarity':{},'scope_note':'Attribution only; not lesion detection, pathology confirmation, or evidence of cancer.'}

def normalize_case(age=None,sex=None,tobacco=None,alcohol=None,lesion_site=None,magnification=None):
    return {'age':age or 'Not provided','sex':sex or 'Not provided','tobacco':tobacco or 'Not provided','alcohol':alcohol or 'Not provided','lesion_site':lesion_site or 'Not provided','magnification':magnification or 'Not provided'}

def xai_retrieval_terms(xai_summary):
    summaries=[]
    for name,item in xai_summary.get('methods',{}).items():
        if item.get('status')=='ok':
            stats=item.get('statistics',{})
            summaries.append(f'{name} attribution summary mean {stats.get("mean",0):.3f} q90 {stats.get("q90",0):.3f}')
    agreement=xai_summary.get('pairwise_similarity',{})
    correlations=[v.get('pearson_correlation') for v in agreement.values() if v.get('pearson_correlation') is not None]
    agreement_text=('attribution map agreement mean Pearson similarity '+str(round(float(np.mean(correlations)),3))) if correlations else 'attribution map agreement unavailable'
    summaries.append(agreement_text)
    return '; '.join(summaries) if summaries else 'No usable attribution maps were generated.'
def build_case_query(prediction_label,oscc_probability,patient_context,user_question,xai_summary):
    parts=['oral squamous cell carcinoma histopathology research evidence',f'classifier predicted class {prediction_label}',f'OSCC sigmoid score {oscc_probability:.6f}',f'fixed classification threshold {CLASSIFICATION_THRESHOLD:.6f}',xai_retrieval_terms(xai_summary)]
    for key,value in patient_context.items():
        if value and value!='Not provided': parts.append(f'{key}: {value}')
    if user_question.strip(): parts.append(f'research question: {user_question.strip()}')
    parts.append('model attribution is not a pathological finding')
    return ' '.join(parts)

def get_oscc_graph_context(retrieval_query):
    terms=[]
    for term in re.findall(r'[a-zA-Z]{4,}',retrieval_query.lower()):
        if term not in {'oral','model','score','research','evidence','attribution','histopathology','threshold','predicted','normal','classification'} and term not in terms: terms.append(term)
    query='''
    MATCH (d:Disease {name:"Oral Squamous Cell Carcinoma"})
    OPTIONAL MATCH (d)-[r]-(x)
    WITH d,r,x,toLower(coalesce(x.name,"")+" "+coalesce(x.title,"")+" "+coalesce(type(r),"")) AS search_text
    WITH d,r,x,reduce(score=0,term IN $terms | score+CASE WHEN term<>"" AND search_text CONTAINS term THEN 1 ELSE 0 END) AS query_overlap
    RETURN d.name AS disease,type(r) AS relationship,labels(x) AS node_labels,
           x.name AS node_name,x.title AS node_title,x.paper_id AS paper_id,
           x.year AS year,query_overlap
    ORDER BY query_overlap DESC,relationship,node_name,node_title LIMIT 50
    '''
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(query,terms=terms[:30])]

def retrieve_literature(query,top_k=5):
    embedding=embedder.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype('float32')
    scores,indices=faiss_index.search(embedding,top_k); results=[]
    for score,idx in zip(scores[0],indices[0]):
        if idx<0 or idx>=len(chunk_metadata): continue
        item=dict(chunk_metadata[idx]); item['similarity']=float(score); results.append(item)
    return results

def connect_literature_to_graph(literature_evidence,graph_context):
    by_paper={}
    for node in graph_context:
        paper_id=node.get('paper_id')
        if paper_id is not None: by_paper.setdefault(str(paper_id),[]).append({k:node.get(k) for k in ('node_labels','node_name','node_title','relationship')})
    linked=[]
    for item in literature_evidence:
        evidence=dict(item); paper_id=evidence.get('paper_id')
        evidence['graph_entity_matches']=by_paper.get(str(paper_id),[]) if paper_id is not None else []
        linked.append(evidence)
    return linked

def save_case_to_graph(case_id,filename,patient_context,prediction_label,oscc_probability,xai_summary,user_question):
    query='''
    MERGE (c:Case {case_id:$case_id})
    SET c.filename=$filename,c.age=$age,c.sex=$sex,c.tobacco=$tobacco,c.alcohol=$alcohol,
        c.lesion_site=$lesion_site,c.magnification=$magnification,c.prediction=$prediction,
        c.confidence=$confidence,c.oscc_probability=$oscc_probability,
        c.classification_threshold=$threshold,c.xai_summary_json=$xai_json,
        c.user_question=$user_question,c.updated_at=$updated_at
    MERGE (p:Prediction {case_id:$case_id})
    SET p.label=$prediction,p.confidence=$confidence,p.oscc_probability=$oscc_probability,
        p.classification_threshold=$threshold,p.xai_summary_json=$xai_json,p.updated_at=$updated_at
    MERGE (c)-[:HAS_MODEL_PREDICTION]->(p)
    WITH p OPTIONAL MATCH (d:Disease {name:"Oral Squamous Cell Carcinoma"})
    FOREACH (_ IN CASE WHEN $prediction="OSCC" AND d IS NOT NULL THEN [1] ELSE [] END |
        MERGE (p)-[:PREDICTS_CLASS]->(d))
    '''
    confidence=oscc_probability if prediction_label=='OSCC' else 1.-oscc_probability
    params={'case_id':case_id,'filename':filename,**patient_context,'prediction':prediction_label,'confidence':float(confidence),'oscc_probability':float(oscc_probability),'threshold':float(CLASSIFICATION_THRESHOLD),'xai_json':json.dumps(xai_summary,ensure_ascii=False),'user_question':user_question,'updated_at':datetime.now(timezone.utc).isoformat()}
    with neo4j_driver.session(database=NEO4J_DATABASE) as session: session.run(query,**params).consume()

print('Inference, four-method XAI and Graph-RAG helpers are ready.')



In [ ]:
import json
import logging

logger = logging.getLogger("oscc_graphrag")
MAX_REPORT_PROMPT_CHARS = 60000

REPORT_SYSTEM_INSTRUCTION = """
You are a research-support assistant for an oral cancer histopathology project.
Use only supplied case data, retrieved graph context and retrieved literature.

Rules:
- The classifier output is a model prediction, not a clinical probability or diagnosis.
- The OSCC sigmoid score is a model score, not probability that a patient has cancer.
- Heatmaps indicate model attribution only; they do not prove a lesion, cancer, or pathology location.
- Never infer histological features or biomarkers from heatmap statistics.
- Graph relationships and literature are research/population context, not case-specific findings.
- Only cite references supplied in retrieved chunks. Exact paper_id matches may be described as shared identifiers; do not invent graph edges.
- Separate retrieved evidence from generated interpretation and disclose missing evidence.
- Never invent patient information, references, or findings. Do not recommend treatment.
- Explicitly state this is research/academic support, not a clinical diagnosis.

Use these sections: Case Summary; Model Prediction; XAI Attribution; Knowledge-Graph Context; Retrieved Literature; Integrated Research Interpretation; Limitations.
"""


def _bound_prompt_value(value, max_string=5000, max_items=40):
    if isinstance(value, str):
        if len(value) > max_string:
            return value[:max_string] + " [truncated for report context size]"
        return value
    if isinstance(value, dict):
        return {str(k): _bound_prompt_value(v, max_string, max_items) for k, v in list(value.items())[:max_items]}
    if isinstance(value, (list, tuple)):
        return [_bound_prompt_value(v, max_string, max_items) for v in list(value)[:max_items]]
    if value is None or isinstance(value, (int, float, bool)):
        return value
    return str(value)


def generate_research_report(case_id, filename, patient_context, prediction_label,
                             oscc_probability, threshold, xai_summary, user_question,
                             graph_context, retrieval_query, literature_evidence):
    payload = {
        "case_id": case_id,
        "filename": filename,
        "user_question": user_question,
        "patient_context": patient_context,
        "classifier": {
            "predicted_class": prediction_label,
            "oscc_sigmoid_score": float(oscc_probability),
            "fixed_threshold": float(threshold),
        },
        "xai_summary": xai_summary,
        "knowledge_graph_context": graph_context,
        "retrieval_query": retrieval_query,
        "retrieved_literature": literature_evidence,
    }
    payload = _bound_prompt_value(payload)
    prompt = f"""
Prepare a grounded research explanation for this OSCC image case. Separate model output, attribution, retrieved graph/literature evidence, and synthesis. If a source is missing, say so.

Structured source data (no raw image or heatmap arrays):
{json.dumps(payload, indent=2, ensure_ascii=False)}
"""
    prompt_chars = len(REPORT_SYSTEM_INSTRUCTION) + len(prompt)
    if prompt_chars > MAX_REPORT_PROMPT_CHARS:
        raise ValueError(f"Report context is {prompt_chars} characters; limit is {MAX_REPORT_PROMPT_CHARS}.")

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": REPORT_SYSTEM_INSTRUCTION},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_completion_tokens=4096,
        reasoning_effort="low",
        include_reasoning=False,
    )
    if not getattr(response, "choices", None):
        raise RuntimeError("Groq returned no completion choices.")
    choice = response.choices[0]
    content = getattr(getattr(choice, "message", None), "content", None)
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError(
            "Groq returned no final text content; check response finish_reason and service logs."
        )
    if getattr(choice, "finish_reason", None) == "length":
        raise RuntimeError("Groq stopped before completing the report (finish_reason=length).")
    logger.info(
        "Groq report generated case_id=%s model=%s prompt_chars=%s finish_reason=%s",
        case_id, GROQ_MODEL, prompt_chars, getattr(choice, "finish_reason", None),
    )
    return content.strip()

def make_report_error_response(case_id, exc):
    # Called inside the API exception handler so logger.exception records the traceback.
    logger.exception("Groq report generation failed case_id=%s model=%s", case_id, GROQ_MODEL)
    return {
        "status": "error",
        "provider": "Groq",
        "model": GROQ_MODEL,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "error": {
            "code": type(exc).__name__,
            "message": "Report generation failed. See backend logs for the request traceback.",
        },
        "text": None,
    }

print("Grounded Groq report generator defined.")


In [ ]:
from fastapi.responses import HTMLResponse,FileResponse
from fastapi import FastAPI,UploadFile,File,Form,HTTPException
from fastapi.middleware.cors import CORSMiddleware
import re

MAX_UPLOAD_BYTES=20*1024*1024
app=FastAPI(title='OSCC GraphRAG Backend',version='2.0')
app.add_middleware(CORSMiddleware,allow_origins=['*'],allow_credentials=False,allow_methods=['*'],allow_headers=['*'])

@app.get('/health')
def health():
    return {'status':'ok','cnn':model is not None,'faiss':faiss_index is not None,'neo4j':neo4j_driver is not None,'groq':groq_client is not None,'groq_model':GROQ_MODEL,'model':'EfficientNetB0','threshold':CLASSIFICATION_THRESHOLD,'xai_methods':XAI_METHOD_NAMES}

@app.get('/heatmaps/{case_id}/{filename}')
def get_heatmap(case_id:str,filename:str):
    allowed={stem+suffix for stem in XAI_FILE_NAMES.values() for suffix in ('.png','_overlay.png')}
    if not re.fullmatch(r'CASE-[A-F0-9]{10}',case_id) or filename not in allowed: raise HTTPException(status_code=404,detail='Heatmap not found')
    path=ARTIFACT_DIR/case_id/filename
    if not path.is_file(): raise HTTPException(status_code=404,detail='Heatmap not found')
    return FileResponse(path,media_type='image/png',filename=filename)

@app.post('/analyze')
async def analyze(image:UploadFile=File(...),age:str=Form(''),sex:str=Form(''),tobacco:str=Form(''),alcohol:str=Form(''),lesion_site:str=Form(''),magnification:str=Form(''),research_question:str=Form('')):
    try: image_bytes=await image.read(MAX_UPLOAD_BYTES+1)
    finally: await image.close()
    if not image_bytes: raise HTTPException(status_code=400,detail='Uploaded image is empty')
    if len(image_bytes)>MAX_UPLOAD_BYTES: raise HTTPException(status_code=413,detail='Image exceeds 20 MB limit')
    if image.content_type and not image.content_type.startswith('image/'): raise HTTPException(status_code=415,detail='Upload must be an image')
    filename=Path(image.filename or 'uploaded_image').name
    try: prediction_result=predict_image(image_bytes)
    except Exception as exc: raise HTTPException(status_code=400,detail=f'Image could not be decoded or scored ({type(exc).__name__})')
    probability=prediction_result['oscc_probability']; prediction_label=prediction_result['prediction']
    patient_context=normalize_case(age,sex,tobacco,alcohol,lesion_site,magnification)
    case_id='CASE-'+uuid.uuid4().hex[:10].upper()
    xai_summary=generate_xai(prediction_result['image_tensor'],prediction_result['rgb_image'],case_id,probability)
    retrieval_query=build_case_query(prediction_label,probability,patient_context,research_question or '',xai_summary)
    errors=[]
    try: graph_context=get_oscc_graph_context(retrieval_query)
    except Exception as exc:
        graph_context=[]; errors.append({'stage':'graph_retrieval','error':type(exc).__name__})
    try:
        literature_evidence=retrieve_literature(retrieval_query,top_k=5)
        literature_evidence=connect_literature_to_graph(literature_evidence,graph_context)
    except Exception as exc:
        literature_evidence=[]; errors.append({'stage':'literature_retrieval','error':type(exc).__name__})
    try:
        save_case_to_graph(case_id,filename,patient_context,prediction_label,probability,xai_summary,research_question or '')
        graph_saved=True
    except Exception as exc:
        graph_saved=False; errors.append({'stage':'case_graph_write','error':type(exc).__name__})
    try:
        report_text=generate_research_report(case_id,filename,patient_context,prediction_label,probability,CLASSIFICATION_THRESHOLD,xai_summary,research_question or '',graph_context,retrieval_query,literature_evidence)
        report={'status':'ok','provider':'Groq','model':GROQ_MODEL,'generated_at':datetime.now(timezone.utc).isoformat(),'text':report_text}
    except Exception as exc:
        report=make_report_error_response(case_id,exc)
        errors.append({'stage':'report_generation','error':report['error']['code']})
    confidence=float(prediction_result['confidence'])
    return {'case_id':case_id,'filename':filename,'patient_context':patient_context,'research_question':research_question or '',
            'prediction':prediction_label,'class_index':int(CLASS_MAPPING[prediction_label]),'oscc_probability':float(probability),'threshold':float(CLASSIFICATION_THRESHOLD),
            'decision_rule':'OSCC when OSCC sigmoid score >= 0.850266; otherwise Normal',
            'confidence':confidence,'confidence_percent':round(confidence*100,2),
            'xai':xai_summary,'graph_saved':graph_saved,'graph_context':graph_context,'retrieval_query':retrieval_query,
            'literature_evidence':literature_evidence,'errors':errors,
            'scope_note':'The model score is not clinical probability. Heatmaps indicate model attribution only and do not prove pathology. Graph and literature provide research context, not specimen-specific confirmation.',
            'report':report}

print('FastAPI routes: GET /health, POST /analyze, GET /heatmaps/{case_id}/{filename}, GET /docs, GET /')

# ------------------------------------------------------------
# Built-in frontend
# Polished single-page frontend served by FastAPI.
# No separate frontend project is required.
@app.get("/", response_class=HTMLResponse)
def home():
    return HTML_PAGE

HTML_PAGE = '\\\n<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1.0">\n<title>OSCC GraphRAG | Research Console</title>\n<style>\n:root{--bg:#07111f;--panel:#0d1b2e;--panel2:#10243b;--line:#1e3652;--text:#eaf2ff;--muted:#8fa6bf;--accent:#36d1dc;--accent2:#6c7cff;--good:#35d07f;--warn:#ffbd59;--danger:#ff6b7a;--shadow:0 24px 70px rgba(0,0,0,.28)}\n*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:radial-gradient(circle at 15% 0%,#123254 0,transparent 34%),radial-gradient(circle at 90% 10%,#24235b 0,transparent 30%),var(--bg);color:var(--text);font-family:Inter,ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif}.wrap{max-width:1180px;margin:auto;padding:28px 20px 60px}.top{display:flex;justify-content:space-between;align-items:center;gap:20px;margin-bottom:24px}.brand{display:flex;align-items:center;gap:13px}.logo{width:46px;height:46px;border-radius:14px;background:linear-gradient(135deg,var(--accent),var(--accent2));display:grid;place-items:center;font-size:23px;font-weight:900;box-shadow:0 10px 30px rgba(54,209,220,.22)}.brand h1{font-size:18px;margin:0}.brand p{margin:3px 0 0;color:var(--muted);font-size:12px}.status-pill{border:1px solid #27506c;background:#0c2034;padding:9px 12px;border-radius:999px;font-size:12px;color:#b9d3e9}.dot{display:inline-block;width:7px;height:7px;background:var(--good);border-radius:50%;margin-right:7px;box-shadow:0 0 12px var(--good)}\n.hero{position:relative;overflow:hidden;padding:38px;border:1px solid var(--line);border-radius:26px;background:linear-gradient(135deg,rgba(16,36,59,.95),rgba(10,24,43,.82));box-shadow:var(--shadow);margin-bottom:22px}.hero:after{content:"";position:absolute;width:280px;height:280px;border-radius:50%;right:-90px;top:-120px;background:rgba(54,209,220,.12);filter:blur(8px)}.eyebrow{color:var(--accent);font-size:12px;font-weight:800;letter-spacing:1.5px;text-transform:uppercase}.hero h2{font-size:clamp(28px,5vw,48px);line-height:1.05;margin:10px 0 12px;max-width:760px}.hero p{color:var(--muted);max-width:760px;line-height:1.65;margin:0}.tech{display:flex;flex-wrap:wrap;gap:8px;margin-top:20px}.tech span{padding:7px 10px;border:1px solid var(--line);border-radius:999px;background:#0a1829;color:#b8cae0;font-size:11px}\n.layout{display:grid;grid-template-columns:1fr 360px;gap:20px}.card{background:rgba(13,27,46,.94);border:1px solid var(--line);border-radius:22px;box-shadow:var(--shadow);padding:22px}.card h3{margin:0 0 5px;font-size:18px}.sub{color:var(--muted);font-size:12px;margin:0 0 18px}.drop{border:1.5px dashed #376080;border-radius:18px;padding:30px 18px;text-align:center;background:linear-gradient(180deg,#0c1c2f,#0a1727);cursor:pointer;transition:.2s}.drop:hover,.drop.drag{border-color:var(--accent);background:#0d2238}.upload-icon{font-size:34px;margin-bottom:10px}.drop strong{display:block;font-size:15px}.drop small{display:block;color:var(--muted);margin-top:7px}.drop input{display:none}.preview{display:none;margin-top:14px;grid-template-columns:130px 1fr;gap:14px;padding:12px;border:1px solid var(--line);border-radius:15px;background:#091728}.preview img{width:130px;height:110px;object-fit:cover;border-radius:10px}.filemeta{display:flex;flex-direction:column;justify-content:center;gap:5px}.filemeta b{font-size:13px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.filemeta span{font-size:11px;color:var(--muted)}.grid{display:grid;grid-template-columns:repeat(2,1fr);gap:13px;margin-top:16px}.field label{display:block;font-size:11px;color:#9db2c8;margin:0 0 6px;font-weight:700}.field input,.field select{width:100%;padding:12px 13px;border:1px solid var(--line);border-radius:11px;background:#091728;color:var(--text);outline:none}.field input:focus,.field select:focus{border-color:var(--accent);box-shadow:0 0 0 3px rgba(54,209,220,.08)}.full{grid-column:1/-1}.actions{display:flex;gap:10px;margin-top:16px}.primary,.secondary{border:0;border-radius:12px;padding:13px 16px;font-weight:800;cursor:pointer}.primary{flex:1;color:#04131d;background:linear-gradient(135deg,var(--accent),#7ce7df);box-shadow:0 12px 25px rgba(54,209,220,.14)}.primary:disabled{opacity:.55;cursor:wait}.secondary{background:#10243b;color:#bcd0e4;border:1px solid var(--line)}.notice{margin-top:13px;padding:11px 13px;border-radius:11px;background:#0b1b2d;border:1px solid var(--line);color:#91a8bf;font-size:11px;line-height:1.5}.side-stat{display:grid;grid-template-columns:1fr 1fr;gap:10px}.stat{padding:14px;border:1px solid var(--line);border-radius:15px;background:#0a1829}.stat b{display:block;font-size:17px;margin-top:4px}.stat span{font-size:10px;color:var(--muted)}.steps{display:flex;flex-direction:column;gap:11px;margin-top:18px}.step{display:flex;gap:11px;align-items:flex-start;color:#8ea4bb;font-size:12px}.step .n{width:25px;height:25px;border-radius:8px;border:1px solid var(--line);display:grid;place-items:center;flex:none}.step.active{color:#d9e8f8}.step.active .n{border-color:var(--accent);color:var(--accent)}\n#status{min-height:20px;margin-top:13px;font-size:12px;color:var(--muted)}.loader{display:none;height:5px;background:#10243b;border-radius:999px;overflow:hidden;margin-top:8px}.loader i{display:block;width:35%;height:100%;background:linear-gradient(90deg,var(--accent),var(--accent2));animation:load 1.1s infinite}.results{display:none;margin-top:20px}.result-head{display:flex;justify-content:space-between;gap:15px;align-items:center;flex-wrap:wrap}.case{font-size:11px;color:var(--muted)}.badges{display:flex;flex-wrap:wrap;gap:8px}.badge{padding:8px 11px;border-radius:999px;border:1px solid var(--line);background:#0a1829;font-size:11px;color:#c6d7e8}.badge strong{color:white}.result-grid{display:grid;grid-template-columns:270px 1fr;gap:18px;margin-top:18px}.confidence{display:grid;place-items:center;min-height:245px;border:1px solid var(--line);border-radius:18px;background:#091728}.ring{--p:0deg;width:150px;height:150px;border-radius:50%;display:grid;place-items:center;background:conic-gradient(var(--accent) var(--p),#19314a 0deg);position:relative}.ring:after{content:"";position:absolute;width:116px;height:116px;background:#091728;border-radius:50%}.ring div{position:relative;z-index:1;text-align:center}.ring b{display:block;font-size:25px}.ring span{font-size:10px;color:var(--muted)}.prediction{margin-top:12px;font-size:13px;font-weight:800}.tabs{display:flex;gap:7px;border-bottom:1px solid var(--line);padding-bottom:8px}.tab{border:0;background:transparent;color:#8198b0;padding:8px 10px;border-radius:8px;cursor:pointer;font-size:11px;font-weight:700}.tab.active{background:#102b43;color:#cfe9f3}.panel{display:none;padding-top:14px}.panel.active{display:block}.report{white-space:pre-wrap;line-height:1.7;color:#c8d7e7;font-size:13px}.evidence{display:grid;gap:9px}.ev{border:1px solid var(--line);background:#091728;border-radius:12px;padding:12px}.ev b{font-size:12px}.ev p{margin:5px 0 0;color:#8fa7bd;font-size:11px;line-height:1.5}.jsonbox{white-space:pre-wrap;background:#07111f;border:1px solid var(--line);padding:13px;border-radius:12px;color:#9db5cb;font-size:11px;line-height:1.55;max-height:360px;overflow:auto}.footer{margin-top:24px;color:#617991;text-align:center;font-size:10px}.toast{position:fixed;right:18px;bottom:18px;background:#102b43;border:1px solid #31536e;color:#d9ebf7;padding:12px 15px;border-radius:12px;box-shadow:var(--shadow);display:none;font-size:12px;z-index:10}@keyframes load{0%{transform:translateX(-120%)}100%{transform:translateX(330%)}}@media(max-width:900px){.layout,.result-grid{grid-template-columns:1fr}.side{order:-1}.side-stat{grid-template-columns:repeat(4,1fr)}}@media(max-width:620px){.wrap{padding:16px 12px 40px}.hero{padding:25px}.grid{grid-template-columns:1fr}.full{grid-column:auto}.top{align-items:flex-start}.status-pill{display:none}.side-stat{grid-template-columns:1fr 1fr}.preview{grid-template-columns:90px 1fr}.preview img{width:90px;height:85px}.actions{flex-direction:column}}\n</style>\n</head>\n<body>\n<div class="wrap">\n  <header class="top"><div class="brand"><div class="logo">◈</div><div><h1>OSCC GraphRAG</h1><p>Research &amp; knowledge intelligence console</p></div></div><div class="status-pill"><span class="dot"></span>Backend ready</div></header>\n  <section class="hero"><div class="eyebrow">AI-assisted histopathology research</div><h2>Explore an OSCC case with vision + knowledge.</h2><p>Upload a histopathology image and combine EfficientNetB0 prediction with retrieved literature, a Neo4j knowledge graph and a Groq-generated research report.</p><div class="tech"><span>EfficientNetB0</span><span>FAISS Retrieval</span><span>Neo4j Graph</span><span>Groq LLM</span></div></section>\n  <div class="layout">\n    <main>\n      <section class="card"><h3>New analysis</h3><p class="sub">Provide the image and optional case context.</p>\n        <form id="analysisForm">\n          <label class="drop" id="dropZone"><input id="image" name="image" type="file" accept="image/*" required><div class="upload-icon">⇧</div><strong>Drop histopathology image here</strong><small>or click to browse · PNG, JPG, JPEG</small></label>\n          <div class="preview" id="preview"><img id="previewImg"><div class="filemeta"><b id="fileName">—</b><span id="fileSize">—</span><span>Image selected and ready for analysis</span></div></div>\n          <div class="grid">\n            <div class="field"><label>AGE</label><input id="age" name="age" placeholder="e.g. 55"></div>\n            <div class="field"><label>SEX</label><select id="sex" name="sex"><option value="">Select</option><option>Male</option><option>Female</option><option>Other</option></select></div>\n            <div class="field"><label>TOBACCO</label><input id="tobacco" name="tobacco" placeholder="Yes / No"></div>\n            <div class="field"><label>ALCOHOL</label><input id="alcohol" name="alcohol" placeholder="Yes / No"></div>\n            <div class="field"><label>LESION SITE</label><input id="lesion_site" name="lesion_site" placeholder="e.g. Buccal mucosa"></div>\n            <div class="field"><label>MAGNIFICATION</label><input id="magnification" name="magnification" placeholder="e.g. 40x"></div>\n          </div>\n          <div class="actions"><button class="primary" id="submitBtn" type="submit">Analyze case →</button><button class="secondary" id="resetBtn" type="button">Reset</button></div>\n          <div class="loader" id="loader"><i></i></div><div id="status"></div>\n          <div class="notice">Research support only. Model confidence is not a clinical probability, and the retrieved literature does not prove that a specific histological feature is present in the uploaded image.</div>\n        </form>\n      </section>\n      <section class="card results" id="resultCard">\n        <div class="result-head"><div><h3>Case analysis</h3><div class="case" id="caseLine">—</div></div><div class="badges" id="badges"></div></div>\n        <div class="result-grid"><div class="confidence"><div><div class="ring" id="ring"><div><b id="confidenceValue">0%</b><span>model confidence</span></div></div><div class="prediction" id="prediction">Prediction: —</div></div></div>\n          <div><div class="tabs"><button class="tab active" data-tab="reportPanel">Research report</button><button class="tab" data-tab="litPanel">Literature</button><button class="tab" data-tab="graphPanel">Knowledge graph</button></div>\n            <div class="panel active" id="reportPanel"><div class="report" id="report"></div><button class="secondary" id="downloadBtn" style="margin-top:12px">Download report</button></div>\n            <div class="panel" id="litPanel"><div class="evidence" id="literature"></div></div>\n            <div class="panel" id="graphPanel"><div class="jsonbox" id="graph"></div></div>\n          </div>\n        </div>\n      </section>\n    </main>\n    <aside class="side"><section class="card"><h3>Pipeline</h3><p class="sub">What happens after you submit</p><div class="side-stat"><div class="stat"><span>VISION</span><b>CNN</b></div><div class="stat"><span>SEARCH</span><b>FAISS</b></div><div class="stat"><span>GRAPH</span><b>Neo4j</b></div><div class="stat"><span>REPORT</span><b>Groq</b></div></div><div class="steps"><div class="step active"><div class="n">1</div><div><b>Image inference</b><br>EfficientNetB0 produces the model output.</div></div><div class="step active"><div class="n">2</div><div><b>Evidence retrieval</b><br>Relevant research chunks are retrieved from FAISS.</div></div><div class="step active"><div class="n">3</div><div><b>Graph context</b><br>Neo4j contributes structured case knowledge.</div></div><div class="step active"><div class="n">4</div><div><b>Research synthesis</b><br>Groq turns the supplied evidence into a report.</div></div></div></section></aside>\n  </div>\n  <div class="footer">OSCC GraphRAG · Academic/research prototype · Not a diagnostic device</div>\n</div><div class="toast" id="toast"></div>\n<script>\nconst $=id=>document.getElementById(id); const form=$(\'analysisForm\'), file=$(\'image\'), drop=$(\'dropZone\'), preview=$(\'preview\'); let latest=null;\nfunction toast(msg){const t=$(\'toast\');t.textContent=msg;t.style.display=\'block\';setTimeout(()=>t.style.display=\'none\',2600)}\nfunction formatBytes(n){if(!n)return \'0 B\';const u=[\'B\',\'KB\',\'MB\',\'GB\'];let i=0;while(n>=1024&&i<3){n/=1024;i++}return n.toFixed(i?1:0)+\' \'+u[i]}\nfunction showFile(f){if(!f)return; $(\'fileName\').textContent=f.name;$(\'fileSize\').textContent=formatBytes(f.size);$(\'previewImg\').src=URL.createObjectURL(f);preview.style.display=\'grid\'}\nfile.addEventListener(\'change\',()=>showFile(file.files[0]));[\'dragenter\',\'dragover\'].forEach(e=>drop.addEventListener(e,x=>{x.preventDefault();drop.classList.add(\'drag\')}));[\'dragleave\',\'drop\'].forEach(e=>drop.addEventListener(e,x=>{x.preventDefault();drop.classList.remove(\'drag\')}));drop.addEventListener(\'drop\',e=>{const f=e.dataTransfer.files[0];if(f&&f.type.startsWith(\'image/\')){file.files=e.dataTransfer.files;showFile(f)}else toast(\'Please drop an image file.\')});\n$(\'resetBtn\').onclick=()=>{form.reset();preview.style.display=\'none\';$(\'resultCard\').style.display=\'none\';$(\'status\').textContent=\'\';latest=null};\ndocument.querySelectorAll(\'.tab\').forEach(b=>b.onclick=()=>{document.querySelectorAll(\'.tab\').forEach(x=>x.classList.remove(\'active\'));document.querySelectorAll(\'.panel\').forEach(x=>x.classList.remove(\'active\'));b.classList.add(\'active\');$(b.dataset.tab).classList.add(\'active\')});\nfunction esc(v){return String(v??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]))}\nfunction literatureHTML(items){if(!Array.isArray(items)||!items.length)return \'<div class="ev"><b>No literature evidence returned</b><p>The retrieval step returned no displayable records.</p></div>\';return items.map((x,i)=>{if(typeof x===\'string\')return `<div class="ev"><b>Evidence ${i+1}</b><p>${esc(x)}</p></div>`;const title=x.title||x.paper_title||x.name||`Evidence ${i+1}`;const body=x.text||x.content||x.chunk||x.snippet||x.abstract||JSON.stringify(x);return `<div class="ev"><b>${esc(title)}</b><p>${esc(body)}</p></div>`}).join(\'\')}\nform.addEventListener(\'submit\',async e=>{e.preventDefault();if(!file.files[0])return toast(\'Select a histopathology image first.\');const fd=new FormData();fd.append(\'image\',file.files[0]);[\'age\',\'sex\',\'tobacco\',\'alcohol\',\'lesion_site\',\'magnification\'].forEach(id=>fd.append(id,$(id).value));$(\'submitBtn\').disabled=true;$(\'loader\').style.display=\'block\';$(\'status\').textContent=\'Running image inference → retrieval → graph → report…\';$(\'resultCard\').style.display=\'none\';try{const r=await fetch(\'/analyze\',{method:\'POST\',body:fd});let d;try{d=await r.json()}catch(_){throw new Error(\'Server returned an invalid response.\')}if(!r.ok)throw new Error(d.detail||\'Analysis failed.\');latest=d;const pct=Number(d.confidence_percent??(Number(d.confidence||0)*100));$(\'confidenceValue\').textContent=pct.toFixed(1)+\'%\';$(\'ring\').style.setProperty(\'--p\',Math.max(0,Math.min(100,pct))*3.6+\'deg\');$(\'prediction\').textContent=\'Prediction: \'+(d.prediction||\'N/A\');$(\'caseLine\').textContent=(d.case_id||\'CASE\')+\' · \'+(d.filename||\'image\');$(\'badges\').innerHTML=`<span class="badge">Model <strong>EfficientNetB0</strong></span><span class="badge">LLM <strong>Groq</strong></span>`;$(\'report\').textContent=d.report?.text||\'No report returned.\';$(\'literature\').innerHTML=literatureHTML(d.literature_evidence);$(\'graph\').textContent=JSON.stringify(d.graph_context??{},null,2);$(\'resultCard\').style.display=\'block\';$(\'status\').textContent=\'Analysis completed successfully.\';toast(\'Case analysis complete.\');setTimeout(()=>$(\'resultCard\').scrollIntoView({behavior:\'smooth\',block:\'start\'}),80)}catch(err){$(\'status\').textContent=\'Error: \'+err.message;toast(\'Analysis failed.\')}finally{$(\'submitBtn\').disabled=false;$(\'loader\').style.display=\'none\'}});\n$(\'downloadBtn\').onclick=()=>{if(!latest)return;const text=`OSCC GraphRAG Research Report\\n\\nCase: ${latest.case_id||\'\'}\\nPrediction: ${latest.prediction||\'\'}\\nConfidence: ${latest.confidence_percent??\'\'}%\\n\\n${latest.report?.text||\'\'}\\n\\nScope note:\\n${latest.scope_note||\'\'}`;const a=document.createElement(\'a\');a.href=URL.createObjectURL(new Blob([text],{type:\'text/plain\'}));a.download=(latest.case_id||\'OSCC_case\')+\'_report.txt\';a.click();URL.revokeObjectURL(a.href)};\n</script></body></html>\n'

print("Enhanced frontend route created successfully.")
print("Website URL: PUBLIC_BACKEND_URL + '/'")

def replace_html_once(old,new,label):
    global HTML_PAGE
    if HTML_PAGE.count(old)!=1: raise RuntimeError(f'{label} anchor count: {HTML_PAGE.count(old)}')
    HTML_PAGE=HTML_PAGE.replace(old,new,1)
replace_html_once('<span>EfficientNetB0</span><span>FAISS Retrieval</span>','<span>EfficientNetB0</span><span>Four XAI attributions</span><span>FAISS Retrieval</span>','header tech')
replace_html_once('<div class="field"><label>MAGNIFICATION</label><input id="magnification" name="magnification" placeholder="e.g. 40x"></div>\n          </div>','<div class="field"><label>MAGNIFICATION</label><input id="magnification" name="magnification" placeholder="e.g. 40x"></div>\n            <div class="field full"><label>RESEARCH QUESTION (OPTIONAL)</label><input id="research_question" name="research_question" placeholder="Question to guide retrieval"></div>\n          </div>','research question field')
replace_html_once('<span>model confidence</span>','<span>OSCC model score</span>','score label')
replace_html_once('<button class="tab active" data-tab="reportPanel">Research report</button>','<button class="tab" data-tab="xaiPanel">XAI attribution</button><button class="tab active" data-tab="reportPanel">Research report</button>','XAI tab')
replace_html_once('<div class="panel active" id="reportPanel">','<div class="panel" id="xaiPanel"><p class="sub">Attribution overlays reflect model behavior only; they do not prove pathology.</p><div class="xai-grid" id="xaiHeatmaps"></div></div>\n            <div class="panel active" id="reportPanel">','XAI panel')
replace_html_once('.evidence{display:grid;gap:9px}','.evidence{display:grid;gap:9px}.xai-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px}.xai-card{border:1px solid var(--line);background:#091728;border-radius:12px;padding:10px}.xai-card img{width:100%;display:block;border-radius:8px;margin-top:8px}.xai-card p{color:#8fa7bd;font-size:11px;line-height:1.5;margin:7px 0 0}@media(max-width:620px){.xai-grid{grid-template-columns:1fr}}','XAI CSS')
xai_js="""function xaiHTML(xai){const methods=xai?.methods||{};const cards=Object.entries(methods).map(function(pair){const name=pair[0],item=pair[1];if(item.status!==\"ok\")return '<div class=\\\"xai-card\\\"><b>'+esc(name)+' · unavailable</b><p>'+esc(item.error||\"Attribution unavailable\")+'</p></div>';const s=item.statistics||{};return '<div class=\\\"xai-card\\\"><b>'+esc(name)+'</b><img src=\\\"'+esc(item.overlay_url||item.heatmap_url)+'\\\" alt=\\\"'+esc(name)+' attribution overlay\\\"><p>Normalized mean '+Number(s.mean||0).toFixed(3)+' · q90 '+Number(s.q90||0).toFixed(3)+' · area fraction '+Number(s.fraction_at_or_above_0_5||0).toFixed(3)+'</p></div>'}).join(\"\");return cards||\"<div class=\\\"ev\\\">No attribution methods returned.</div>\"}"""
replace_html_once('</script>',xai_js+'\n</script>','XAI renderer')
replace_html_once("['age','sex','tobacco','alcohol','lesion_site','magnification'].forEach","['age','sex','tobacco','alcohol','lesion_site','magnification','research_question'].forEach",'question submission')
replace_html_once('Running image inference → retrieval → graph → report…','Running image inference → four attributions → retrieval → graph → report…','status text')
replace_html_once("const pct=Number(d.confidence_percent??(Number(d.confidence||0)*100));$('confidenceValue').textContent=pct.toFixed(1)+'%';$('ring').style.setProperty('--p',Math.max(0,Math.min(100,pct))*3.6+'deg');$('prediction').textContent='Prediction: '+(d.prediction||'N/A');","const pct=Number(d.oscc_probability||0)*100;$('confidenceValue').textContent=pct.toFixed(1)+'%';$('ring').style.setProperty('--p',Math.max(0,Math.min(100,pct))*3.6+'deg');$('prediction').textContent='Prediction: '+(d.prediction||'N/A')+' · threshold '+Number(d.threshold).toFixed(6);",'prediction display')
replace_html_once("$('report').textContent=d.report?.text||'No report returned.';","$('report').textContent=d.report?.text||'No report returned.';$('xaiHeatmaps').innerHTML=xaiHTML(d.xai);",'XAI result render')


reference_renderer="""function literatureHTMLWithReferences(items){if(!Array.isArray(items)||!items.length)return '<div class=\"ev\"><b>No literature evidence returned</b><p>Retrieval returned no displayable records.</p></div>';return items.map(function(x,i){if(typeof x==='string')return '<div class=\"ev\"><b>Evidence '+(i+1)+'</b><p>'+esc(x)+'</p></div>';const title=x.title||x.paper_title||x.name||('Evidence '+(i+1));const body=x.text||x.content||x.chunk||x.snippet||x.abstract||JSON.stringify(x);const ref=x.citation||x.reference||x.paper_id||'';const year=x.year?(' · '+x.year):'';const rawUrl=x.url||x.source_url||x.doi_url||'';const safeUrl=(String(rawUrl).startsWith('https://')||String(rawUrl).startsWith('http://'))?String(rawUrl):'';const link=safeUrl?'<a href=\"'+esc(safeUrl)+'\" target=\"_blank\" rel=\"noopener noreferrer\">Open source</a>':'';return '<div class=\"ev\"><b>'+esc(title)+'</b><p>'+esc(body)+'</p>'+(ref?'<p>Reference: '+esc(ref)+esc(year)+'</p>':'')+link+'</div>'}).join('')}"""
replace_html_once('</script>',reference_renderer+'\n</script>','literature references renderer')
replace_html_once("$('literature').innerHTML=literatureHTML(d.literature_evidence);","$('literature').innerHTML=literatureHTMLWithReferences(d.literature_evidence);",'literature reference display')



replace_html_once("$('report').textContent=d.report?.text||'No report returned.';$('xaiHeatmaps').innerHTML=xaiHTML(d.xai);", "const reportReady=d.report?.status==='ok'&&typeof d.report?.text==='string'&&d.report.text.trim().length>0;$('report').textContent=reportReady?d.report.text:('Research report generation failed. '+(d.report?.error?.message||'Check backend logs.'));$('downloadBtn').disabled=!reportReady;$('xaiHeatmaps').innerHTML=xaiHTML(d.xai);", 'report status display')
replace_html_once("$('status').textContent='Analysis completed successfully.';toast('Case analysis complete.');", "$('status').textContent=reportReady?'Analysis completed successfully.':'Image analysis completed, but the report could not be generated. Check backend logs.';toast(reportReady?'Case analysis complete.':'Report generation failed.');", 'report status feedback')
replace_html_once("$('downloadBtn').onclick=()=>{if(!latest)return;", "$('downloadBtn').onclick=()=>{if(!latest||latest.report?.status!=='ok'||!latest.report?.text){toast('No generated report is available to download.');return;}", 'download guard')
replace_html_once("Confidence: ${latest.confidence_percent??''}%", "OSCC score: ${Number(latest.oscc_probability).toFixed(6)}\nThreshold: ${Number(latest.threshold).toFixed(6)}", 'download score header')


In [ ]:
# Clean old FastAPI/ngrok processes before starting

from pyngrok import ngrok

ngrok.kill()

subprocess.run(
    ["bash", "-lc", "fuser -k 8000/tcp || true"],
    capture_output=True,
    text=True
)

print("Cleanup complete.")

In [ ]:
# Start FastAPI server

import uvicorn

server_thread = threading.Thread(
    target=lambda: uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    ),
    daemon=True
)

server_thread.start()
time.sleep(3)

print("FastAPI server started.")
print("Local health: http://127.0.0.1:8000/health")
print("Local docs:   http://127.0.0.1:8000/docs")

In [ ]:
# Test local backend before exposing it publicly

import requests

r = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=20
)

print("HTTP status:", r.status_code)
print(json.dumps(r.json(), indent=2))

In [ ]:
from google.colab import userdata
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig
import requests

# 1. Read token from Colab Secrets
NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")

if not NGROK_TOKEN:
    raise ValueError(
        "NGROK_AUTHTOKEN not found. Check Colab Secrets and notebook access."
    )

# 2. Verify FastAPI
health = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)
health.raise_for_status()

print("FastAPI is running:", health.status_code)

# 3. Stop any previous ngrok agent
ngrok.kill()

# 4. Use a fresh configuration file
config = PyngrokConfig(
    config_path="/content/ngrok_config.yml"
)

# 5. Configure authentication
ngrok.set_auth_token(
    NGROK_TOKEN,
    pyngrok_config=config
)

# 6. Start tunnel using the automatically assigned dev domain
tunnel = ngrok.connect(
    addr=8000,
    proto="http",
    pyngrok_config=config
)

PUBLIC_BACKEND_URL = tunnel.public_url

print("\nBackend URL:", PUBLIC_BACKEND_URL)
print("Swagger Docs:", PUBLIC_BACKEND_URL + "/docs")
print("Health:", PUBLIC_BACKEND_URL + "/health")
print("Analyze:", PUBLIC_BACKEND_URL + "/analyze")

print("\nActive tunnels:")
print(ngrok.get_tunnels())

## Built-in Website + Backend URLs

The embedded frontend is served by FastAPI. After the tunnel cell runs, use the printed PUBLIC_BACKEND_URL for the website, /docs, /health, and /analyze. Heatmap URLs returned by /analyze are served from the same backend under /heatmaps/...


In [ ]:
# Final public health check

r = requests.get(
    PUBLIC_BACKEND_URL + "/health",
    timeout=20
)

print("HTTP status:", r.status_code)
print(json.dumps(r.json(), indent=2))

In [ ]:
# Optional end-to-end /analyze test.
# Change IMAGE_PATH if your test image is elsewhere.

IMAGE_PATH = "/content/OSCC_400x_450.jpg"

if os.path.exists(IMAGE_PATH):
    with open(IMAGE_PATH, "rb") as f:
        response = requests.post(
            PUBLIC_BACKEND_URL + "/analyze",
            files={
                "image": (
                    os.path.basename(IMAGE_PATH),
                    f,
                    "image/jpeg"
                )
            },
            data={
                "age": "",
                "sex": "",
                "tobacco": "",
                "alcohol": "",
                "lesion_site": "",
                "magnification": ""
            },
            timeout=180
        )

    print("Status:", response.status_code)
    print(response.text)
else:
    print("Test image not found.")
    print("Set IMAGE_PATH to an image in /content and run this cell again.")

In [ ]:
from pyngrok import ngrok

# Stop any previous local ngrok process
ngrok.kill()

# Reuse the already configured authentication
tunnel = ngrok.connect(
    addr=8000,
    proto="http"
)

print("Public URL:", tunnel.public_url)
print("Swagger Docs:", tunnel.public_url + "/docs")
print("Active tunnels:", ngrok.get_tunnels())